In [1]:
from datetime import datetime, timedelta
import numpy as np
import pycountry
import json
from IPython.display import display, Markdown
from matplotlib import pyplot as plt
from matplotlib.pyplot import cm

from emu_renewal.inputs import DATA_PATH, TEXT_DATE_FORMAT, \
    get_worldbank_national_pop, get_country_mobility, get_standard_targets, \
    get_country_vacc_data, get_country_vars, find_relevant_vars
from emu_renewal.utils import get_row_proportions

In [2]:
def create_single_country_preplot(country, end_plot_time):
    start_plot_time = datetime(2020, 1, 1)

    # Gather common data
    iso3 = pycountry.countries.lookup(country).alpha_3
    country_name = pycountry.countries.lookup(country).name
    pop = get_worldbank_national_pop(iso3)
    hosp_indicator, hosp_colour = ("Weekly new hospital admissions", "black") if iso3 in initial_countries["admissions"] else ("Daily hospital occupancy", "red")
    cases_target, hosp_target, deaths_target, seroprev_target, init_data = get_standard_targets(iso3, datetime(2020, 1, 1), end_plot_time, 50, hosp_indicator)

    # Create figure
    fig, axes = plt.subplots(7, 1, figsize=[12, 15], sharex=True)
    fig.suptitle(country_name, fontsize=18, y=1.0)

    # Plot cases
    axes[0].plot(cases_target.index, cases_target, linewidth=0, marker=".")
    axes[0].set_title("cases")

    # Plot hospitalisations
    axes[1].plot(hosp_target.index, hosp_target, linewidth=0, marker=".", color=hosp_colour)
    axes[1].set_title("hospitalisations")

    # Plot deaths
    death_start_threshold = 2e-6
    per_capita_deaths = deaths_target / pop
    axes[2].plot(deaths_target.index, deaths_target, linewidth=0, marker=".")
    start_time = per_capita_deaths.index[per_capita_deaths.gt(death_start_threshold)].min()
    axes[2].axvline(start_time)
    axes[2].set_title("deaths")

    # Plot seroprevalence
    axes[3].plot(seroprev_target.index, seroprev_target, linewidth=0, marker=".")
    axes[3].set_title("seroprevalence")
    axes[3].set_ylim([0.0, 1.0])

    # Plot mobility
    mobility = get_country_mobility(iso3)
    axes[4].plot(mobility.index, mobility)

    # Plot variants
    var_country_name = pycountry.countries.lookup(iso3).official_name if iso3 in ["CZE"] else pycountry.countries.lookup(iso3).name
    var_data = get_country_vars(var_country_name)
    relevant_vars = find_relevant_vars(var_data, end_plot_time - timedelta(270), 10)
    select_var_data = var_data[relevant_vars]
    var_props = get_row_proportions(select_var_data)
    axes[5].stackplot(var_props.index, var_props.T, colors=cm.turbo(np.linspace(0.0, 1.0, len(var_props.columns))), labels=var_props.columns)
    axes[5].legend(ncols=2)
    pre_alpha_vars = ["20A.EU1"] if iso3 == "LTU" else ["20A.EU1", "20A.EU2"]
    eu_prop = select_var_data[pre_alpha_vars].sum(axis=1) / select_var_data.sum(axis=1)
    axes[5].plot(eu_prop.index, eu_prop, linewidth=0, marker=".", markerfacecolor="w", markersize=10)

    # Plot vaccination
    vacc_data = get_country_vacc_data(country)
    cov_threshold = 0.05
    axes[6].plot(vacc_data.index, vacc_data)
    cov_time = vacc_data[vacc_data.gt(cov_threshold * 100)].idxmin()
    axes[6].axvline(cov_time)
    axes[6].set_ylim(0.0, 100.0)

    # Tidy figure
    axes[0].set_xlim([start_plot_time, end_plot_time])
    plt.setp(axes[-1].xaxis.get_majorticklabels(), rotation=70)
    fig.tight_layout()

    # Report text results
    display(Markdown(f"### {country_name}\nISO3: {iso3}\n"))
    display(Markdown(f"Population: {round(pop / 1e6, 3)} million\n"))
    display(Markdown(f"Date at which {death_start_threshold * 1e6} deaths per million per day exceeded: {start_time.strftime(TEXT_DATE_FORMAT)}"))
    display(Markdown(f"Date at which {round(cov_threshold * 100)}% vaccination coverage exceeded: {cov_time.strftime(TEXT_DATE_FORMAT)}"))
    display(Markdown(f"Hospitalisation indicator: {hosp_indicator}"))
    
    return fig

In [3]:
initial_countries = json.load(open(DATA_PATH / f"config/countries.json", "r"))
all_countries = initial_countries["admissions"] + initial_countries["occupancy"]

for c in all_countries:
    figure = create_single_country_preplot(c, datetime(2021, 12, 31))

TypeError: get_standard_targets() takes 4 positional arguments but 5 were given